In [0]:
#!pip install lseg-data
%pip install lseg-data --quiet

In [0]:
# Run these commands in a separate terminal or notebook cell to set up secrets:
# This cell is for reference only - run these commands once to store credentials securely

# Create secret scope (run once)
# databricks secrets create-scope refinitiv_scope

# Add secrets (run once for each secret)
# databricks secrets put-secret refinitiv_scope app_key --string-value "fgdssgdf"
# databricks secrets put-secret refinitiv_scope username --string-value "nvaldez@tec.mx"
# databricks secrets put-secret refinitiv_scope password --string-value "cK"

print("\nAfter running the above commands, use the cell below to retrieve secrets securely.")
print("Note: These commands should be run in Databricks CLI or using the Secrets API.")

In [0]:
# Create secret scope and store credentials securely
# Run this cell ONCE to set up your secrets

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Create secret scope
try:
    w.secrets.create_scope(scope="refinitiv_scope")
    print("✓ Secret scope 'refinitiv_scope' created successfully")
except Exception as e:
    print(f"Note: {e} (scope may already exist)")

# Store credentials
try:
    w.secrets.put_secret(
        scope="refinitiv_scope",
        key="app_key",
        string_value="b"
    )
    print("✓ app_key stored")
    
    w.secrets.put_secret(
        scope="refinitiv_scope",
        key="username",
        string_value="nvaldez@tec.mx"
    )
    print("✓ username stored")
    
    w.secrets.put_secret(
        scope="refinitiv_scope",
        key="password",
        string_value="cKccccccccccccccccc"
    )
    print("✓ password stored")
    
    print("\n✓ All credentials stored securely in Databricks Secrets!")
except Exception as e:
    print(f"Error storing secrets: {e}")

In [0]:
# Fetch Apple stock prices for the last year using ld API
import lseg.data as ld
from datetime import datetime, timedelta

# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")


In [0]:
import lseg.data as ld

# Check version
print("lseg.data version:", ld.__version__)

# Check if we need to set the session as default
print("\nChecking session management methods:")
if hasattr(ld, 'set_default_session'):
    print("  ✓ ld.set_default_session() exists")
if hasattr(ld.session, 'set_default'):
    print("  ✓ ld.session.set_default() exists")

# List session-related methods
print("\nSession-related methods in ld:")
for attr in sorted(dir(ld)):
    if 'session' in attr.lower() and not attr.startswith('_'):
        print(f"  - {attr}")

In [0]:

# Calculate date range for last year
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

# Fetch historical data using ld.get_history (not get_data)
df = ld.get_history(

    universe="AAPL.O",
    fields=["TR.PriceClose.date", "TR.PriceClose", "TR.PriceOpen", "TR.PriceHigh", "TR.PriceLow", "TR.Volume"],
    interval="1D",
    start=start_date,
    end=end_date
)

# Display the result
display(df)